In [3]:
import torch 
import torch.nn as nn
import torch.optim as optim
import torchvision

In [4]:
import os
print(os.listdir("./animals/cat")[:5])
print(os.listdir("./animals/dog")[:5])

['00013-4122619886.png', '00006-4122619879.png', '00436-200124746.png', '00077-200124387.png', '00457-200124767.png']
['00708-3846168870.png', '00758-3846168920.png', '00656-3846168818.png', '00733-3846168895.png', '00802-3846168964.png']


In [5]:
# fiding ipynb_checkpoints as it will create error going forward, as folders should only have images not .ipynb files
import os

for root, dirs, files in os.walk("./animals"):
    if ".ipynb_checkpoints" in dirs:
        print("Found:", os.path.join(root, ".ipynb_checkpoints"))

In [6]:
import shutil, os

for root, dirs, files in os.walk("./animals"):
    if ".ipynb_checkpoints" in dirs:
        path = os.path.join(root, ".ipynb_checkpoints")
        shutil.rmtree(path)
        print("Removed:", path)

In [7]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((64, 64)),   # cat/dog images are inconsistent sizes, CIFAR10 was already 32x32
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

full_dataset = ImageFolder(root="./animals", transform=transform)

# split into train/test since ImageFolder doesn't give you train/test separately
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
trainset, testset = random_split(full_dataset, [train_size, test_size])

In [8]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testoader = DataLoader(testset, batch_size=64)

### Build CNN

In [9]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

        )
        
        self.fc_layers = nn.Sequential(
            nn.Linear(8*8*128, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)

        return x

In [10]:
model = CNN()

In [11]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters())

In [12]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0
    for images, labels in trainloader:
        optimizer.zero_grad()
        outputs = model(images).squeeze(1) # [batch] instead of [batch, 1]
        labels = labels.float() # ImageFolder gives int labels by default
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_training_loss += loss.item()
    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=0.6179075470337501
epoch=2/10 & loss=0.5769845430667584
epoch=3/10 & loss=0.4045859162624066
epoch=4/10 & loss=0.29747809355075544
epoch=5/10 & loss=0.23507115588738367
epoch=6/10 & loss=0.18550235606156862
epoch=7/10 & loss=0.1404227287723468
epoch=8/10 & loss=0.08988398686051369
epoch=9/10 & loss=0.08925818651914597
epoch=10/10 & loss=0.057780153332994535


In [14]:
# Evaluation
correct_labels = 0
total_labels = 0
model.eval()

with torch.no_grad():
    for images, labels in testoader:
        outputs = model(images).squeeze(1)
        predicted = (torch.sigmoid(outputs) > 0.5).long()
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels}")

accuracy = 0.955


In [17]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix

all_preds, all_labels = [], []
model.eval()

with torch.no_grad():
    for images, labels in testoader:
        outputs = model(images).squeeze(1)
        predicted = (torch.sigmoid(outputs) > 0.5).long()
        all_preds.extend(predicted.tolist())
        all_labels.extend(labels.tolist())

print("precision: ", precision_score(all_labels, all_preds))
print("recall: ", recall_score(all_labels, all_preds))
print("confusion matrix: ", confusion_matrix(all_labels, all_preds))

precision:  0.9285714285714286
recall:  0.978494623655914
confusion matrix:  [[100   7]
 [  2  91]]
